Для запуска нужен файл `.env` (см. `.env.example`), в котором заданы:  
- токен телеграмм бота (`TELEGRAM_BOT_TOKEN`)  
- чат id в переписке с ботом (`TELEGRAM_CHAT_ID`)  
- токен api T-bank (`TBANK_API_TOKEN`)  
- номер основного счета в T-bank (`TBANK_ACCOUNT_NUMBER`)  
- номер договора на овернайт с T-bank (`OVERNIGHT_AGREEMENT_NUMBER`)  
- пути к сертификату и ключу mTLS (`CERTFILE`, `KEYFILE`)  

Переменные загружаются через `python-dotenv`: `pip install python-dotenv`

In [137]:
import os

from dotenv import load_dotenv

# читает переменные из файла .env в каталоге ноутбука
load_dotenv()

tt = os.environ['TBANK_API_TOKEN']
tg = os.environ['TELEGRAM_BOT_TOKEN']
chat_id = os.environ['TELEGRAM_CHAT_ID']
main_account = os.environ['TBANK_ACCOUNT_NUMBER']
agreement_number = os.environ['OVERNIGHT_AGREEMENT_NUMBER']
certfile = os.environ['CERTFILE']
keyfile = os.environ['KEYFILE']

In [ ]:
import http.client
import json
import ssl

def get_ballance(token=tt, account=main_account):
    conn = http.client.HTTPSConnection("business.tbank.ru")
    payload = ''
    headers = {
      'Accept': 'application/json',
      'Authorization': f'Bearer {token}'
    }
    conn.request("GET", "/openapi/api/v4/bank-accounts", payload, headers)
    res = conn.getresponse()
    data = res.read().decode("utf-8")
    accounts_data = json.loads(data)
    counter = {}
    for i in accounts_data:
        counter[i['accountNumber']] = i['balance']['balance']
    
    return(counter[account])


In [ ]:
def transfer_to_overnight(amount, agreement_number,
                          certfile=certfile,
                          keyfile=keyfile):

    context = ssl.create_default_context()
    context.load_cert_chain(certfile, keyfile)
    conn = http.client.HTTPSConnection("secured-openapi.tbank.ru", context=context)

    amount = str(amount-500)

    path = f"/api/v1/overnight/replenish?agreementNumber={agreement_number}&amount={amount}"

    headers = {
        'Accept': 'application/json',
        'Authorization': f'Bearer {tt}'  # Ваш токен
    }

    payload = ''

    # Отправляем POST-запрос
    conn.request("POST", path, payload, headers)

    # Получаем ответ
    res = conn.getresponse()
    data = res.read()

    return(res.status)

In [ ]:
def send_telegram_message(tg, chat_id, message_text):
    """
    Отправляет сообщение в Telegram через бота.
    
    Args:
        bot_token (str): Токен вашего бота от @BotFather
        chat_id (str или int): ID чата для отправки сообщения
        message_text (str): Текст сообщения для отправки
    
    Returns:
        bool: True если сообщение отправлено успешно, False в случае ошибки
    """
    try:
        # Создаем соединение с сервером Telegram
        conn = http.client.HTTPSConnection("api.telegram.org")
        
        # Формируем данные для отправки
        url = f"/bot{tg}/sendMessage"
        params = {
            'chat_id': chat_id,
            'text': message_text,
            'parse_mode': 'HTML'  # Можно использовать HTML-разметку
        }
        
        # Отправляем POST-запрос
        conn.request("POST", url, 
                     body=json.dumps(params), 
                     headers={'Content-Type': 'application/json'})
        
        # Получаем ответ
        response = conn.getresponse()
        
        # Закрываем соединение
        conn.close()
    except Exception as e:
        return False

In [122]:
amount = get_ballance()
ok = f'Овернайт пополнен на {amount} руб.'
oops = 'С овернайтом что-то не так, проверь'

if amount >= 150000:
    result = transfer_to_overnight(amount)
    send_telegram_message(tg, chat_id, ok if result == 200 else oops)
